In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.sparse import csr_matrix # Para manejar matrices dispersas eficientemente
from sklearn.metrics.pairwise import cosine_similarity # Para calcular similitud
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning) # Ignorar warnings de sparsidad si aparecen


In [16]:

# --- Parámetros (Ajusta según sea necesario) ---
TRANSACTION_DATA_PATH = '../data/raw/data_recomendacion.csv'  # Ruta al archivo de transacciones
PRODUCTS_DATA_PATH = '../data/raw/productos.csv'  # Ruta al catálogo de productos
# --- Parámetros ---
NUM_RECOMMENDATIONS = 5    # Número de recomendaciones finales por cliente

print("Parámetros configurados:")
print(f"  Archivo Transacciones: {TRANSACTION_DATA_PATH}")
print(f"  Archivo Productos: {PRODUCTS_DATA_PATH}")
print(f"  Num Recomendaciones: {NUM_RECOMMENDATIONS}")



Parámetros configurados:
  Archivo Transacciones: ../data/raw/data_recomendacion.csv
  Archivo Productos: ../data/raw/productos.csv
  Num Recomendaciones: 5


In [17]:
print(f"\nCargando transacciones: {TRANSACTION_DATA_PATH}")
transactions_df = pd.read_csv(TRANSACTION_DATA_PATH, parse_dates=['date_sale'])
products_df = pd.read_csv(PRODUCTS_DATA_PATH, usecols=["codigo_unico", "description", "category"])
products_df = products_df.rename(columns={"codigo_unico": "product"})
transactions_df = pd.merge(transactions_df, products_df, how='left', on='product')


Cargando transacciones: ../data/raw/data_recomendacion.csv


In [35]:
excluir_prd = ["BOLSA PLASTICA", "TRANSPORTE DOMICILIO"]
transactions_df = transactions_df[~transactions_df["description"].isin(excluir_prd)]

In [36]:
prod = pd.DataFrame(transactions_df["description"].value_counts(normalize=False)).reset_index()
prod

,description,count
0,TOMATE CHONTO,14117
1,PLATANO,11505
2,ZANAHORIA,9926
3,BANANO CRIOLLO,9119
4,CEBOLLA BLANCA,9058
...,...,...
15667,DESO MEN SPEED STICK ENERGY X2 *50GR,1
15668,CUADERNO COS. MPC B CUADRITOS,1
15669,SACAPUNTAS FANTASMA EN EMPAQ QUIZZ,1
15670,CHOCOLATE 70% MANI-ALMEND-PISTACHO *60GR,1


In [38]:
q50 = prod["count"].quantile(0.50)
q99 = prod["count"].quantile(1)
cond = prod["count"].between(q50, q99, inclusive="both")
prod[cond]

,description,count
0,TOMATE CHONTO,14117
1,PLATANO,11505
2,ZANAHORIA,9926
3,BANANO CRIOLLO,9119
4,CEBOLLA BLANCA,9058
...,...,...
7960,CREM DE LECHE PASTE SEMIENT COLANTA*970G,10
7961,ALMENDRAS NATURALES LA CORUÑA x 200GR,10
7962,MANGO TOMMY PICADO* 300 GR,10
7963,BEBIDA ALMENDRAS DUOPACK TONING*946ML,10


In [8]:
transactions_df["id_client"].nunique(), transactions_df["product"].nunique()

(700, 15805)

In [9]:
user_item_counts = transactions_df.groupby(['id_client', 'product']).size().reset_index(name='purchase_count')
print(f"Interacciones únicas Usuario-Item: {len(user_item_counts)}")

Interacciones únicas Usuario-Item: 251384


In [10]:
# Crear mapeos de IDs a índices numéricos (necesario para matriz dispersa)
user_item_counts['client_id'] = user_item_counts['id_client'].astype('category').cat.codes
user_item_counts['product_id'] = user_item_counts['product'].astype('category').cat.codes


In [12]:
# Guardar los mapeos para poder traducir de vuelta
user_map = dict(enumerate(user_item_counts['id_client'].astype('category').cat.categories))
product_map = dict(enumerate(user_item_counts['product'].astype('category').cat.categories))
user_map_inv = {v: k for k, v in user_map.items()} # ID -> índice
product_map_inv = {v: k for k, v in product_map.items()} # ID -> índice


In [13]:
print(f"Número de usuarios únicos: {len(user_map)}")
print(f"Número de productos únicos en interacciones: {len(product_map)}")


Número de usuarios únicos: 700
Número de productos únicos en interacciones: 15805


In [14]:
# Crear la matriz dispersa CSR (Compressed Sparse Row)
    # Filas: usuarios, Columnas: productos, Valores: conteo de compras
try:
    sparse_user_item = csr_matrix(
        (user_item_counts['purchase_count'],
         (user_item_counts['client_id'], user_item_counts['product_id'])),
        shape=(len(user_map), len(product_map))
    )
    print(f"Matriz Usuario-Item dispersa creada. Forma: {sparse_user_item.shape}")
    # Calcular densidad (qué % de celdas tienen valor)
    sparsity = 1.0 - (sparse_user_item.nnz / (sparse_user_item.shape[0] * sparse_user_item.shape[1]))
    print(f"Densidad de la matriz: {1.0 - sparsity:.4f} (Sparsity: {sparsity:.4f})")
except Exception as e:
    print(f"Error creando la matriz dispersa: {e}")
    sparse_user_item = None

Matriz Usuario-Item dispersa creada. Forma: (700, 15805)
Densidad de la matriz: 0.0227 (Sparsity: 0.9773)


In [15]:
# %% [code]
# === Función para Generar Recomendaciones Item-Item ===

def get_item_item_recs_notebook(client_str_id, sparse_user_item_matrix, similarity_matrix,
                                product_str_to_idx, product_idx_to_str, user_str_to_idx,
                                n_recs=5):
    """
    Genera recomendaciones Item-Item CF para un cliente.
    Maneja tanto la matriz de similitud dispersa como el DataFrame.
    """
    recommendations = {} # {product_idx: score}

    if client_str_id not in user_str_to_idx:
        # print(f"Advertencia: Cliente {client_str_id} no encontrado en datos de interacción.")
        return [] # Cliente no tiene historial registrado

    client_idx = user_str_to_idx[client_str_id]
    client_purchases_sparse = sparse_user_item_matrix[client_idx, :]
    # Indices de productos comprados por el cliente
    purchased_product_indices = client_purchases_sparse.indices
    # Valores de interacción (ej. conteo de compras)
    purchase_values = client_purchases_sparse.data

    if len(purchased_product_indices) == 0:
        # print(f"Advertencia: Cliente {client_str_id} no tiene compras registradas.")
        return []

    # Calcular scores sumando similitudes ponderadas
    for i, purchased_idx in enumerate(purchased_product_indices):
        purchase_count = purchase_values[i] # Conteo de compra del item 'gatillo'

        # Obtener fila de similitud para el item comprado
        if isinstance(similarity_matrix, pd.DataFrame):
             # Asegurarse que el índice exista en el DataFrame de similitud
             purchased_prod_str = product_idx_to_str.get(purchased_idx)
             if purchased_prod_str in similarity_matrix.index:
                 sim_row = similarity_matrix.loc[purchased_prod_str].values # Obtener como numpy array
             else: continue # Producto no en matriz de similitud
        else: # Es una matriz dispersa (o numpy)
             sim_row = similarity_matrix[purchased_idx, :].toarray().flatten()

        # Iterar sobre todas las similitudes de este item
        for similar_item_idx, similarity_score in enumerate(sim_row):
            # Considerar solo items diferentes al actual y con similitud positiva
            if similar_item_idx != purchased_idx and similarity_score > 0:
                # MUY IMPORTANTE: Verificar si el cliente YA compró este item similar
                if similar_item_idx not in purchased_product_indices:
                    # Acumular score ponderado por similitud y conteo de compra del item gatillo
                    current_score = recommendations.get(similar_item_idx, 0)
                    recommendations[similar_item_idx] = current_score + (similarity_score * purchase_count)

    # Ordenar recomendaciones por score descendente
    sorted_recs_idx = sorted(recommendations.items(), key=lambda item: item[1], reverse=True)

    # Convertir índices de producto de vuelta a IDs y tomar top N
    final_recs_list = []
    for product_idx, score in sorted_recs_idx:
         product_str = product_idx_to_str.get(product_idx)
         if product_str:
             final_recs_list.append({'recommended_product': product_str, 'recommendation_score': score})
         if len(final_recs_list) >= n_recs:
             break

    return final_recs_list

In [ ]:
# === Generar Recomendaciones para Clientes Objetivo ===

all_recommendations_list = []
sim_matrix = item_similarity_df if item_similarity_df is not None else item_similarity # Usar DF si existe

if sparse_user_item is not None and sim_matrix is not None and target_clients:
    print(f"\nGenerando {NUM_RECOMMENDATIONS} recomendaciones Item-Item por cliente objetivo...")

    for client in target_clients:
        print(f"  Procesando cliente: {client}")
        recs_list = get_item_item_recs_notebook(
            client,
            sparse_user_item,
            sim_matrix, # Pasar la matriz/DataFrame de similitud
            product_map_inv, # Pasar product_str -> product_idx
            product_map,     # Pasar product_idx -> product_str
            user_map_inv,      # Pasar user_str -> user_idx
            n_recs=NUM_RECOMMENDATIONS
        )
        print(f"    Recomendaciones encontradas: {len(recs_list)}")

In [42]:
df = pd.read_parquet("../data/processed/initial_sales_clean.parquet")

In [43]:
df.head()

,date_sale,id_client,product,amount,invoice_value_with_discount_and_without_iva,domicilio_status
0,2024-01-01,22222222222,93240,1.0,7201.0,False
1,2024-01-01,22222222222,81660,1.0,7201.0,False
2,2024-01-01,32206081,254996,1.0,74440.0,False
3,2024-01-01,32206081,107697,1.0,74440.0,False
4,2024-01-01,32206081,109801,1.0,74440.0,False


In [44]:
df.shape

(51962758, 6)